# 01 — Data Loading and Cleaning

Builds `df_development` (2000-2021, used for modelling) and `df_real_world` (2022-2023, unlabeled, kept for reference only) from three sources:

1. **IHME** — mental health disorder prevalence (local CSV, already filtered to EU countries at extraction time)
2. **World Bank API** — socioeconomic indicators
3. **WHO API** — suicide rate (target variable)

EDA starts in `02_eda.ipynb`, once this notebook has produced `data/processed/df_development.parquet`.

In [1]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd

from src import (
    EU_COUNTRIES_ISO,
    WORLD_BANK_INDICATORS,
    WHO_SUICIDE_INDICATOR,
    load_ihme_data,
    fetch_worldbank_indicators,
    fetch_who_suicide_rates,
    build_master_dataset,
    interpolate_with_trend_extrapolation,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

## Step 1 — IHME mental health prevalence (base dataset)

Read from a local CSV. Replace the path below with the location of your IHME export if it differs.

In [2]:
IHME_CSV_PATH = "../data/raw/IHME-GBD_2023_DATA-b62eec84-1.csv"

df_base = load_ihme_data(IHME_CSV_PATH, min_year=2000)
print(f"IHME base dataset: {df_base.shape[0]} rows, {df_base.shape[1]} columns")
display(df_base.head())

IHME base dataset: 648 rows, 14 columns


,Country,Code,Year,Alcohol use disorders,Alzheimer's disease and other dementias,Anxiety disorders,Attention-deficit/hyperactivity disorder,Autism spectrum disorders,Bipolar disorder,Conduct disorder,Depressive disorders,Drug use disorders,Eating disorders,Schizophrenia
6,Austria,AUT,2000,3191.951091,1132.966544,4024.482195,1175.534494,721.252486,720.168715,395.555429,2408.933392,881.662224,479.043042,340.766068
7,Austria,AUT,2001,3193.762279,1143.969355,4046.938653,1173.433463,723.015405,719.736433,394.027233,2414.230764,875.548127,479.489990,342.045193
8,Austria,AUT,2002,3196.670699,1158.019381,4069.554739,1173.688637,724.906163,717.163393,390.398108,2420.619752,867.476203,479.784032,343.374835
9,Austria,AUT,2003,3201.329520,1163.789032,4102.701811,1174.340827,727.316029,714.040173,385.443904,2425.310908,859.486814,480.160291,344.800007
10,Austria,AUT,2004,3206.603569,1163.505776,4147.044336,1173.529004,729.994711,711.859272,380.592234,2428.414977,853.367328,480.506984,346.150489


## Step 2 — World Bank socioeconomic indicators (API)

One request per indicator (the World Bank API does not support multi-indicator queries), merged onto `df_base` by `Code` + `Year`.

In [3]:
df_features = fetch_worldbank_indicators(
    df_base,
    eu_countries_iso=EU_COUNTRIES_ISO,
    indicators=WORLD_BANK_INDICATORS,
    region="ALL",
    date_range="2000:2026",
)
print(
    f"\nFeature dataset after World Bank merge: {df_features.shape[0]} rows, {df_features.shape[1]} columns"
)
display(df_features.head())

World Bank indicator 'GDP per capita' merged. Rows retrieved: 702.
World Bank indicator 'Unemployment rate (%)' merged. Rows retrieved: 702.
World Bank indicator 'Health expenditure (% GDP)' merged. Rows retrieved: 663.
World Bank indicator 'Population' merged. Rows retrieved: 702.
World Bank indicator 'Gini index' merged. Rows retrieved: 589.
World Bank indicator 'Urban population (%)' merged. Rows retrieved: 702.
World Bank indicator 'Physicians per 1000' merged. Rows retrieved: 616.
World Bank indicator 'Internet users (% of population)' merged. Rows retrieved: 678.

Feature dataset after World Bank merge: 648 rows, 22 columns


,Country,Code,Year,Alcohol use disorders,Alzheimer's disease and other dementias,Anxiety disorders,Attention-deficit/hyperactivity disorder,Autism spectrum disorders,Bipolar disorder,Conduct disorder,Depressive disorders,Drug use disorders,Eating disorders,Schizophrenia,GDP per capita,Unemployment rate (%),Health expenditure (% GDP),Population,Gini index,Urban population (%),Internet users (% of population),Physicians per 100000
0,Austria,AUT,2000,3191.951091,1132.966544,4024.482195,1175.534494,721.252486,720.168715,395.555429,2408.933392,881.662224,479.043042,340.766068,24487.297469,4.687,9.389345,8011566.0,29.0,66.769024,33.730133,387.6
1,Austria,AUT,2001,3193.762279,1143.969355,4046.938653,1173.433463,723.015405,719.736433,394.027233,2414.230764,875.548127,479.489990,342.045193,24430.495983,4.007,9.452435,8042293.0,NaN,66.840816,39.185450,398.9
2,Austria,AUT,2002,3196.670699,1158.019381,4069.554739,1173.688637,724.906163,717.163393,390.398108,2420.619752,867.476203,479.784032,343.374835,26334.862215,4.852,9.591399,8081957.0,NaN,66.888746,36.560000,406.0
3,Austria,AUT,2003,3201.329520,1163.789032,4102.701811,1174.340827,727.316029,714.040173,385.443904,2425.310908,859.486814,480.160291,344.800007,32110.115966,4.779,9.698442,8121423.0,29.5,66.926357,42.700000,414.4
4,Austria,AUT,2004,3206.603569,1163.505776,4147.044336,1173.529004,729.994711,711.859272,380.592234,2428.414977,853.367328,480.506984,346.150489,36614.250653,5.969,9.815324,8171966.0,29.8,66.955861,54.280000,423.2


## Step 3 — WHO suicide rate (target variable, API)

Filtered to both sexes (`SEX_BTSX`) and all age groups (`AGEGROUP_YEARSALL`), then merged onto `df_features`.

In [4]:
df_who = fetch_who_suicide_rates(indicator=WHO_SUICIDE_INDICATOR)

df_development, df_real_world = build_master_dataset(
    df_features, df_who, development_cutoff_year=2021
)

print(
    f"df_development: {df_development.shape[0]} rows (2000-2021, labeled — used for modelling)"
)
print(
    f"df_real_world:  {df_real_world.shape[0]} rows (2022-2023, unlabeled — reference only)"
)
display(df_development.head())

df_development: 594 rows (2000-2021, labeled — used for modelling)
df_real_world:  54 rows (2022-2023, unlabeled — reference only)


,Country,Code,Year,Alcohol use disorders,Alzheimer's disease and other dementias,Anxiety disorders,Attention-deficit/hyperactivity disorder,Autism spectrum disorders,Bipolar disorder,Conduct disorder,Depressive disorders,Drug use disorders,Eating disorders,Schizophrenia,GDP per capita,Unemployment rate (%),Health expenditure (% GDP),Population,Gini index,Urban population (%),Internet users (% of population),Physicians per 100000,Suicide rate
0,Austria,AUT,2000,3191.951091,1132.966544,4024.482195,1175.534494,721.252486,720.168715,395.555429,2408.933392,881.662224,479.043042,340.766068,24487.297469,4.687,9.389345,8011566.0,29.0,66.769024,33.730133,387.6,19.753390
1,Austria,AUT,2001,3193.762279,1143.969355,4046.938653,1173.433463,723.015405,719.736433,394.027233,2414.230764,875.548127,479.489990,342.045193,24430.495983,4.007,9.452435,8042293.0,NaN,66.840816,39.185450,398.9,18.508048
2,Austria,AUT,2002,3196.670699,1158.019381,4069.554739,1173.688637,724.906163,717.163393,390.398108,2420.619752,867.476203,479.784032,343.374835,26334.862215,4.852,9.591399,8081957.0,NaN,66.888746,36.560000,406.0,19.413281
3,Austria,AUT,2003,3201.329520,1163.789032,4102.701811,1174.340827,727.316029,714.040173,385.443904,2425.310908,859.486814,480.160291,344.800007,32110.115966,4.779,9.698442,8121423.0,29.5,66.926357,42.700000,414.4,17.989560
4,Austria,AUT,2004,3206.603569,1163.505776,4147.044336,1173.529004,729.994711,711.859272,380.592234,2428.414977,853.367328,480.506984,346.150489,36614.250653,5.969,9.815324,8171966.0,29.8,66.955861,54.280000,423.2,17.383069


## Descriptive check and missing values

In [5]:
print("Master dataset summary statistics")
display(df_development.describe().T)

print("\nMissing values per column:")
remaining_nulls = df_development.isnull().sum()
print(remaining_nulls)

Master dataset summary statistics


,count,mean,std,min,25%,50%,75%,max
Year,594.0,2.010500e+03,6.349636e+00,2000.000000,2.005000e+03,2.010500e+03,2.016000e+03,2.021000e+03
Alcohol use disorders,594.0,2.640863e+03,6.501545e+02,1468.353858,2.154470e+03,2.638039e+03,3.166213e+03,3.930876e+03
Alzheimer's disease and other dementias,594.0,1.162906e+03,3.505339e+02,615.594485,9.072604e+02,1.109865e+03,1.317172e+03,2.490327e+03
Anxiety disorders,594.0,5.060044e+03,1.514067e+03,2850.328655,3.934236e+03,4.670571e+03,5.770123e+03,1.142191e+04
Attention-deficit/hyperactivity disorder,594.0,7.675910e+02,3.598561e+02,248.480691,4.405874e+02,7.954818e+02,9.792250e+02,1.722195e+03
Autism spectrum disorders,594.0,6.979302e+02,1.188224e+02,497.467873,5.765145e+02,7.303758e+02,7.675203e+02,1.011089e+03
Bipolar disorder,594.0,6.507963e+02,8.939936e+01,488.696921,5.619674e+02,6.689073e+02,7.169895e+02,8.472123e+02
Conduct disorder,594.0,3.712084e+02,5.047307e+01,281.621053,3.309906e+02,3.666408e+02,4.003332e+02,5.418604e+02
Depressive disorders,594.0,3.332831e+03,7.327951e+02,2229.578134,2.753071e+03,3.272819e+03,3.777113e+03,6.698489e+03
Drug use disorders,594.0,8.564169e+02,2.555415e+02,282.936650,6.690521e+02,8.318342e+02,9.782403e+02,1.852197e+03



Missing values per column:
Country                                      0
Code                                         0
Year                                         0
Alcohol use disorders                        0
Alzheimer's disease and other dementias      0
Anxiety disorders                            0
Attention-deficit/hyperactivity disorder     0
Autism spectrum disorders                    0
Bipolar disorder                             0
Conduct disorder                             0
Depressive disorders                         0
Drug use disorders                           0
Eating disorders                             0
Schizophrenia                                0
GDP per capita                               0
Unemployment rate (%)                        0
Health expenditure (% GDP)                   0
Population                                   0
Gini index                                  54
Urban population (%)                         0
Internet users (% of population)

In [6]:
# Inspect rows with any missing value
missing_mask = df_development.isnull().any(axis=1)
display(df_development[missing_mask])

,Country,Code,Year,Alcohol use disorders,Alzheimer's disease and other dementias,Anxiety disorders,Attention-deficit/hyperactivity disorder,Autism spectrum disorders,Bipolar disorder,Conduct disorder,Depressive disorders,Drug use disorders,Eating disorders,Schizophrenia,GDP per capita,Unemployment rate (%),Health expenditure (% GDP),Population,Gini index,Urban population (%),Internet users (% of population),Physicians per 100000,Suicide rate
1,Austria,AUT,2001,3193.762279,1143.969355,4046.938653,1173.433463,723.015405,719.736433,394.027233,2414.230764,875.548127,479.489990,342.045193,24430.495983,4.007,9.452435,8042293.0,NaN,66.840816,39.185450,398.9,18.508048
2,Austria,AUT,2002,3196.670699,1158.019381,4069.554739,1173.688637,724.906163,717.163393,390.398108,2420.619752,867.476203,479.784032,343.374835,26334.862215,4.852,9.591399,8081957.0,NaN,66.888746,36.560000,406.0,19.413281
25,Belgium,BEL,2001,2742.250875,1379.930669,4654.220915,901.854461,707.071240,711.290109,400.061362,3363.006903,858.273779,412.408353,336.135116,23015.071263,6.178,8.149889,10286570.0,NaN,82.335689,31.288396,284.9,22.254924
26,Belgium,BEL,2002,2709.540349,1402.582939,4671.415432,898.653329,708.532961,707.756966,399.714573,3317.005557,856.860728,414.020576,337.067605,25006.191397,6.910,8.318211,10332785.0,NaN,82.650966,46.330000,285.9,20.961368
48,Bulgaria,BGR,2000,2245.755453,826.826633,3820.953719,470.767892,497.467873,488.696921,400.313318,2966.043887,640.549583,165.360053,350.857166,1621.345249,16.218,5.829710,8170172.0,NaN,68.263145,5.370923,344.1,18.220209
50,Bulgaria,BGR,2002,2227.972961,870.063596,3832.161321,461.108960,498.983719,493.099454,382.626700,3036.548108,650.344778,166.563910,355.182145,2093.089677,18.110,7.080986,7837161.0,NaN,69.485916,9.080000,352.2,17.326479
52,Bulgaria,BGR,2004,2176.147227,916.925243,3843.118628,450.503914,501.420831,497.092027,361.137543,3085.421090,660.536958,169.422714,359.926417,3389.860480,12.037,6.890909,7716860.0,NaN,69.919334,18.130000,353.7,14.170709
53,Bulgaria,BGR,2005,2154.318462,943.153092,3847.022823,445.075701,502.518188,498.668748,349.649808,2948.029923,661.374512,170.730414,361.824030,3900.025020,10.083,6.890115,7658972.0,NaN,70.086960,19.970000,366.1,13.242697
74,Croatia,HRV,2002,2769.296804,828.412981,5308.754597,466.711028,548.657069,551.600321,388.786125,2479.821041,796.932090,187.597900,363.248197,6219.560937,15.053,6.143005,4302174.0,NaN,55.674547,17.760000,245.9,19.373197
75,Croatia,HRV,2003,2784.748672,849.868797,5338.649218,463.692106,550.083848,552.986871,383.180519,2486.697198,801.062686,190.134302,364.786805,8189.990593,13.924,6.297825,4303399.0,NaN,55.638696,22.750000,251.2,19.643276


## Missing value imputation

Linear interpolation within each country, chosen because the missing values occur in consecutive year runs rather than scattered individual years.

In [7]:
features_with_nan = df_development.columns[df_development.isnull().any()].tolist()
print(f"Columns with missing values detected: {features_with_nan}")

# df_development = impute_missing_values(df_development, features_with_nan)
df_development = interpolate_with_trend_extrapolation(df_development, features_with_nan)

print(
    remaining_nulls[remaining_nulls > 0]
    if remaining_nulls.any()
    else "None — all clean."
)

print("\nMissing values per column after imputation:")
print(df_development.isnull().sum())
display(df_development.head())

Columns with missing values detected: ['Gini index', 'Physicians per 100000']
Gini index               54
Physicians per 100000     4
dtype: int64

Missing values per column after imputation:
Country                                     0
Code                                        0
Year                                        0
Alcohol use disorders                       0
Alzheimer's disease and other dementias     0
Anxiety disorders                           0
Attention-deficit/hyperactivity disorder    0
Autism spectrum disorders                   0
Bipolar disorder                            0
Conduct disorder                            0
Depressive disorders                        0
Drug use disorders                          0
Eating disorders                            0
Schizophrenia                               0
GDP per capita                              0
Unemployment rate (%)                       0
Health expenditure (% GDP)                  0
Population                

,Country,Code,Year,Alcohol use disorders,Alzheimer's disease and other dementias,Anxiety disorders,Attention-deficit/hyperactivity disorder,Autism spectrum disorders,Bipolar disorder,Conduct disorder,Depressive disorders,Drug use disorders,Eating disorders,Schizophrenia,GDP per capita,Unemployment rate (%),Health expenditure (% GDP),Population,Gini index,Urban population (%),Internet users (% of population),Physicians per 100000,Suicide rate
0,Austria,AUT,2000,3191.951091,1132.966544,4024.482195,1175.534494,721.252486,720.168715,395.555429,2408.933392,881.662224,479.043042,340.766068,24487.297469,4.687,9.389345,8011566.0,29.000000,66.769024,33.730133,387.6,19.753390
1,Austria,AUT,2001,3193.762279,1143.969355,4046.938653,1173.433463,723.015405,719.736433,394.027233,2414.230764,875.548127,479.489990,342.045193,24430.495983,4.007,9.452435,8042293.0,29.166667,66.840816,39.185450,398.9,18.508048
2,Austria,AUT,2002,3196.670699,1158.019381,4069.554739,1173.688637,724.906163,717.163393,390.398108,2420.619752,867.476203,479.784032,343.374835,26334.862215,4.852,9.591399,8081957.0,29.333333,66.888746,36.560000,406.0,19.413281
3,Austria,AUT,2003,3201.329520,1163.789032,4102.701811,1174.340827,727.316029,714.040173,385.443904,2425.310908,859.486814,480.160291,344.800007,32110.115966,4.779,9.698442,8121423.0,29.500000,66.926357,42.700000,414.4,17.989560
4,Austria,AUT,2004,3206.603569,1163.505776,4147.044336,1173.529004,729.994711,711.859272,380.592234,2428.414977,853.367328,480.506984,346.150489,36614.250653,5.969,9.815324,8171966.0,29.800000,66.955861,54.280000,423.2,17.383069


## Save processed data

Saved so downstream notebooks (`02_eda.ipynb` onward) can load directly without re-running the API calls above.

In [8]:
df_development.to_csv("../data/processed/df_development.csv", index=False)
print("Saved: data/processed/df_development.csv")

df_real_world.to_csv("../data/processed/df_real_world.csv", index=False)
print("Saved: data/processed/df_real_world.csv")

Saved: data/processed/df_development.csv
Saved: data/processed/df_real_world.csv
